# Global Configuration

The `ElectionConfig` class contains all parameters that apply to an entire election. This notebook shows which parameters exist, what they do, and how to set them.

For a detailed explanation of the concepts, see [Global Configuration](../../docs/source/en/global_configuration.md).

In [ ]:
import pandas as pd
from ipres import ElectionConfig, ConstituenciesConfig, SuperMajorityMargin, MarginUnit
from ipres.election_config import ConstituencyRepresentation, DrawLotsStrategy

## Required Parameters

### `constituencies_config` — Constituencies

The constituencies are passed as a DataFrame. Each row is a constituency with name and size (number of eligible voters).

In [ ]:
wahlkreise_df = pd.DataFrame({
    'constituency_name': ['North', 'South', 'East', 'West', 'Center'],
    'constituency_size': [80_000, 60_000, 70_000, 90_000, 50_000],
})

cc = ConstituenciesConfig.from_dataframe(wahlkreise_df)
print(f"Number of constituencies: {cc.getNumberOfConstituencies()}")

### `participating_parties` — Parties

A simple list of party names.

In [ ]:
parteien = ['A', 'B', 'C', 'D']

config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
)
print(f"Parties: {config.participating_parties}")

---

## `parliament_majority_margin` — Government Margin

The government margin specifies how much more than 50% the government should have — as a buffer so that the majority is maintained even in case of illness or abstentions.

It can be specified either in **percent** or in **seats**. `ElectionConfig` always converts to both units.

In [ ]:
# Specification in percent: Government needs 55% of seats
config_5pct = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
)

# Specification in seats: Government needs 3 seats more than half
config_3seats = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=SuperMajorityMargin(3, MarginUnit.SEATS),
)

print(f"Parliament seats: {config_5pct.parliamentarySeats}")
print()
print("Specification in percent (5%):")
print(f"  Majority threshold: {config_5pct.getParliamentMajorityPercent():.1f}%")
print(f"  Majority threshold: {config_5pct.getParliamentMajoritySeats()} seats")
print()
print("Specification in seats (3 seats):")
print(f"  Majority threshold: {config_3seats.getParliamentMajorityPercent():.1f}%")
print(f"  Majority threshold: {config_3seats.getParliamentMajoritySeats()} seats")

## `constituency_representation` — Constituency Representation

This parameter determines which parties are allocated constituencies, and has a direct impact on the **size of parliament**.

- **`ENTIRE_PARLIAMENT`** *(default)*: All parties receive constituencies proportional to their number of seats. The parliament size is simply calculated as `2 × number of constituencies`.

- **`GOVERNING_MAJORITY`**: Only the governing parties receive constituencies. So that each constituency is represented in the governing faction, parliament must be larger: The total number of parliament seats is chosen such that the governing majority comprises exactly `2 × number of constituencies` seats.

In [ ]:
margin = SuperMajorityMargin(5.0, MarginUnit.PERCENT)  # 55% threshold

config_ep = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=margin,
    constituency_representation=ConstituencyRepresentation.ENTIRE_PARLIAMENT,
)

config_gm = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=margin,
    constituency_representation=ConstituencyRepresentation.GOVERNING_MAJORITY,
)

n = cc.getNumberOfConstituencies()
print(f"{'Constituencies:':<38} {n}")
print(f"{'':38} ENTIRE_PARLIAMENT   GOVERNING_MAJORITY")
print(f"{'Parliament seats':38} {config_ep.parliamentarySeats:<20} {config_gm.parliamentarySeats}")
print(f"{'Government majority (seats)':38} {config_ep.getParliamentMajoritySeats():<20} {config_gm.getParliamentMajoritySeats()}")
print(f"{'Government majority (%)':38} {config_ep.getParliamentMajorityPercent():<20.1f} {config_gm.getParliamentMajorityPercent():.1f}")
print(f"{'Government seats = 2 × constituencies':38} {config_ep.getParliamentMajoritySeats() == 2*n!s:<20} {config_gm.getParliamentMajoritySeats() == 2*n}")

With `GOVERNING_MAJORITY`, parliament is larger because the governing majority must comprise exactly `2 × constituencies` seats. With `ENTIRE_PARLIAMENT`, this property does not hold — there the parliament size is simply fixed.

### Impact of Government Margin on Parliament Size (only with `GOVERNING_MAJORITY`)

The larger the government margin, the larger parliament must be:

In [ ]:
rows = []
for abstand_pct in [0, 2, 5, 10, 15, 20]:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=parteien,
        parliament_majority_margin=SuperMajorityMargin(abstand_pct, MarginUnit.PERCENT),
        constituency_representation=ConstituencyRepresentation.GOVERNING_MAJORITY,
    )
    rows.append({
        'Government margin (%)': abstand_pct,
        'Majority threshold (%)': cfg.getParliamentMajorityPercent(),
        'Parliament seats': cfg.parliamentarySeats,
        'Government seats': cfg.getParliamentMajoritySeats(),
    })

pd.DataFrame(rows).set_index('Government margin (%)')

---

## `draw_lots_strategy` — Tie Resolution

When a tie in seat allocation cannot be clearly resolved, this strategy decides:

- **`RANDOM`** *(default)*: Random decision (influenced by `seed`).
- **`MARGINAL_LEAD`**: The marginal vote difference is treated as a random outcome — the party that happened to receive slightly more votes wins.

In [ ]:
config_random = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    draw_lots_strategy=DrawLotsStrategy.RANDOM,
)

config_marginal_lead = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

print(f"RANDOM:        {config_random.draw_lots_strategy}")
print(f"MARGINAL_LEAD: {config_marginal_lead.draw_lots_strategy}")

---

## `seed` — Reproducibility

With `seed`, the random generator can be set to a fixed seed value so that simulations are reproducible. The default value `None` means that the seed value is chosen randomly.

In [ ]:
# Without seed: different result each time for random decisions
config_no_seed = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
)

# With seed: reproducible
config_seeded = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    seed=42,
)

print(f"No seed: {config_no_seed.seed}")
print(f"With seed:  {config_seeded.seed}")

---

## `ballot_majority_margin` — Ballot Threshold

The `ballot_majority_margin` is the minimum margin above 50% a contestant must reach in a **single voting round** to win that round. It is independent of `parliament_majority_margin`, which governs the seat distribution in parliament.

Like `parliament_majority_margin`, it can be specified in **percent** or **seats**. Default: 2% (i.e. 52%).

In [ ]:
config_ballot = ElectionConfig(
    constituencies_config=cc,
    participating_parties=parteien,
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    ballot_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
)

print(f"Ballot threshold:     {config_ballot.getBallotMajorityPercent():.1f}%  (contestant must reach this share to win a round)")
print(f"Parliament threshold: {config_ballot.getParliamentMajorityPercent():.1f}%  (government must hold this seat share in parliament)")

---

## `language` — Output Language

The `language` parameter controls the language for all display output: table captions, column headers, chart titles, and number formatting.

- **`Language.DE`** *(default)*: German output.
- **`Language.EN`**: English output.

In [ ]:
from ipres.election_config import Language

config_de = ElectionConfig(constituencies_config=cc, participating_parties=parteien, language=Language.DE)
config_en = ElectionConfig(constituencies_config=cc, participating_parties=parteien, language=Language.EN)

print(f"German:  {config_de.language}")
print(f"English: {config_en.language}")

---

## `seat_distribution_method` — Seat Distribution Method

This setting determines the proportional seat distribution method used when evaluating an election.

- **`SAINTE_LAGUE`** *(default)*: Sainte-Laguë method (also known as Webster method). Does not systematically favour any party size.
- **`D_HONDT`**: D'Hondt method. Tends to favour larger parties.
- **`HARE_NIEMEYER`**: Hare-Niemeyer method (largest remainder method). Tends to favour smaller parties.

The evaluator classes (`SeatDistributor`, `ElectionEvaluator`) can override this value for a specific evaluation.

In [ ]:
from ipres.election_config import SeatDistributionMethod

for method in SeatDistributionMethod:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=parteien,
        seat_distribution_method=method,
    )
    print(f"seat_distribution_method = {cfg.seat_distribution_method}")

---

## `quota_correction_strategy` — Quota Correction Strategy

Seat distribution produces integer quotas for constituency allocation (seats ÷ 2). Because the sum of these quotas does not always equal the number of constituencies exactly, a correction step is applied. This setting determines the strategy for that correction.

- **`FAVOR_LARGE_PARTIES`** *(default)*: Extra constituencies go to the largest parties.
- **`FAVOR_SMALL_PARTIES`**: Extra constituencies go to the smallest parties.
- **`PROPORTIONAL`**: Extra constituencies are distributed proportionally to seat count.
- **`PROPORTIONAL_REVERSED`**: Like `PROPORTIONAL`, but in reverse order.
- **`RANDOM`**: Random allocation of extra constituencies.
- **`NEGOTIATED`**: Distribution is determined by a callback.

The evaluator classes (`ConstituencyCountDeterminer`, `ElectionEvaluator`) can override this value for a specific evaluation.

In [ ]:
from ipres.election_config import QuotaCorrectionStrategy

for strategy in [QuotaCorrectionStrategy.FAVOR_LARGE_PARTIES, QuotaCorrectionStrategy.FAVOR_SMALL_PARTIES]:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=parteien,
        quota_correction_strategy=strategy,
    )
    print(f"quota_correction_strategy = {cfg.quota_correction_strategy}")

---

## `constituency_allocation_method` — Constituency Allocation Algorithm

This parameter determines the algorithm used to assign constituencies to parties once the number of constituencies per party has been established.

- **`OPTIMAL`** *(default)*: Optimal assignment that maximises total utility.
- **`GREEDY`**: Greedy algorithm — fast, but not guaranteed to be optimal.
- **`STABLE_MATCHING`**: Stable matching via Gale-Shapley.

The evaluator classes (`ConstituencyAssigner`, `ElectionEvaluator`) can override this value for a specific evaluation.

In [ ]:
from ipres.allocation import ConstituencyAllocationMethod

for method in ConstituencyAllocationMethod:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=parteien,
        constituency_allocation_method=method,
    )
    print(f"constituency_allocation_method = {cfg.constituency_allocation_method}")

---

## Summary

| Parameter                        | Type                           | Default               | Description                                                                            |
|----------------------------------|--------------------------------|-----------------------|----------------------------------------------------------------------------------------|
| `constituencies_config`          | `ConstituenciesConfig`         | —                     | Constituencies (name, size)                                                            |
| `participating_parties`          | `list[str]`                    | —                     | Participating parties                                                                  |
| `parliament_majority_margin`     | `SuperMajorityMargin`          | 5%                    | Government margin in % or seats                                                        |
| `ballot_majority_margin`         | `SuperMajorityMargin`          | 2%                    | Ballot threshold in % or seats                                                         |
| `constituency_representation`    | `ConstituencyRepresentation`   | `ENTIRE_PARLIAMENT`   | Who gets constituencies?                                                               |
| `draw_lots_strategy`             | `DrawLotsStrategy`             | `RANDOM`              | Tie resolution                                                                         |
| `seed`                           | `int \| None`                  | `None`                | Seed value for random generator                                                        |
| `language`                       | `Language`                     | `Language.DE`         | Output language for tables and charts                                                  |
| `seat_distribution_method`       | `SeatDistributionMethod`       | `SAINTE_LAGUE`        | Default seat distribution method; evaluator classes can override                       |
| `quota_correction_strategy`      | `QuotaCorrectionStrategy`      | `FAVOR_LARGE_PARTIES` | Default quota correction strategy; evaluator classes can override                      |
| `constituency_allocation_method` | `ConstituencyAllocationMethod` | `OPTIMAL`             | Default constituency allocation algorithm; evaluator classes can override              |

### Derived Properties

| Property                          | Description                                 |
|-----------------------------------|---------------------------------------------|
| `parliamentarySeats`              | Total number of parliament seats            |
| `getParliamentMajoritySeats()`    | Seats for government majority               |
| `getParliamentMajorityPercent()`  | Majority threshold in %                     |
| `governmentMajorityMarginSeats`   | Margin in seats (converted if necessary)    |
| `governmentMajorityMarginPercent` | Margin in % (converted if necessary)        |